In [3]:
!pip install catboost lightgbm -q


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, confusion_matrix
)

import joblib
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

In [26]:
os.makedirs("../models/artifacts", exist_ok=True)
os.makedirs("../models/trained_models", exist_ok=True)

In [3]:
df = pd.read_csv("../data/Data.csv", na_values=["", "NA", "NaN"], keep_default_na=True)
df = df.fillna(0)   # ensures no NaNs remain

C:\Users\Raj Bharmani\AppData\Local\Temp\ipykernel_17480\2382404254.py:1: DtypeWarning: Columns (9,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/Data.csv", na_values=["", "NA", "NaN"], keep_default_na=True)


In [4]:
def unique_value_distribution(df):
    print("Unique value count per column:\n")
    for col in df.columns:
        unique_vals = df[col].nunique()
        print(f"{col}: {unique_vals} unique values and Dtype: {df[col].dtype}")

In [5]:
unique_value_distribution(df)

Unique value count per column:

BeneID: 138556 unique values and Dtype: object
ClaimID: 558211 unique values and Dtype: object
Provider: 5410 unique values and Dtype: object
InscClaimAmtReimbursed: 438 unique values and Dtype: int64
AttendingPhysician: 82064 unique values and Dtype: object
OperatingPhysician: 35316 unique values and Dtype: object
OtherPhysician: 46458 unique values and Dtype: object
ClmAdmitDiagnosisCode: 4099 unique values and Dtype: object
DeductibleAmtPaid: 17 unique values and Dtype: float64
DiagnosisGroupCode: 1407 unique values and Dtype: object
ClmDiagnosisCode_1: 10451 unique values and Dtype: object
ClmDiagnosisCode_2: 5301 unique values and Dtype: object
ClmDiagnosisCode_3: 4757 unique values and Dtype: object
ClmDiagnosisCode_4: 4360 unique values and Dtype: object
ClmDiagnosisCode_5: 3971 unique values and Dtype: object
ClmDiagnosisCode_6: 3608 unique values and Dtype: object
ClmDiagnosisCode_7: 3389 unique values and Dtype: object
ClmDiagnosisCode_8: 3071 

In [6]:
df = df.drop(columns=["BeneID", "ClaimID", "Provider","AdmissionPeriod"])

In [7]:
df.shape

(558211, 49)

In [8]:
X = df.drop(columns=["PotentialFraud"])
y = df["PotentialFraud"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

In [10]:
class FrequencyEncoder:
    def __init__(self, threshold=100):
        self.threshold = threshold
        self.freq_maps = {}
        self.high_cardinality_cols = []

    def fit(self, X):
        for col in X.columns:
            if X[col].nunique() > self.threshold:
                self.high_cardinality_cols.append(col)
                self.freq_maps[col] = X[col].value_counts().to_dict()
        return self

    def transform(self, X):
        X_transformed = X.copy()
        for col in self.high_cardinality_cols:
            if col in X_transformed.columns:
                X_transformed[col] = X_transformed[col].map(self.freq_maps[col]).fillna(0)
        return X_transformed

    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [11]:
print("Applying frequency encoding...")
freq_encoder = FrequencyEncoder(threshold=100)
X_train_encoded = freq_encoder.fit_transform(X_train_encoded)
X_test_encoded = freq_encoder.transform(X_test_encoded)

print(f"High cardinality columns encoded: {len(freq_encoder.high_cardinality_cols)}")
print(f"Columns: {freq_encoder.high_cardinality_cols}")

joblib.dump(freq_encoder, '../models/artifacts/frequency_encoder.pkl')
print("Frequency encoder saved!")


Applying frequency encoding...
High cardinality columns encoded: 24
Columns: ['InscClaimAmtReimbursed', 'AttendingPhysician', 'OperatingPhysician', 'OtherPhysician', 'ClmAdmitDiagnosisCode', 'DiagnosisGroupCode', 'ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3', 'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6', 'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9', 'ClmDiagnosisCode_10', 'ClmProcedureCode_1', 'ClmProcedureCode_2', 'ClmProcedureCode_3', 'County', 'IPAnnualReimbursementAmt', 'IPAnnualDeductibleAmt', 'OPAnnualReimbursementAmt', 'OPAnnualDeductibleAmt']
Frequency encoder saved!


In [13]:
categorical_columns = X_train_encoded.select_dtypes(include=['object']).columns
label_encoders = {}

for col in tqdm(categorical_columns, desc="Label encoding remaining categorical columns"):
    le = LabelEncoder()
    X_train_encoded[col] = le.fit_transform(X_train_encoded[col].astype(str))
    X_test_encoded[col] = le.transform(X_test_encoded[col].astype(str))
    label_encoders[col] = le


target_encoder = LabelEncoder()
y_train_encoded = target_encoder.fit_transform(y_train)
y_test_encoded = target_encoder.transform(y_test)

# Save label encoders
joblib.dump(label_encoders, '../models/artifacts/label_encoders.pkl')
joblib.dump(target_encoder, '../models/artifacts/target_encoder.pkl')
print("Label encoders saved!")

Label encoding remaining categorical columns: 0it [00:00, ?it/s]

Label encoders saved!


In [14]:
X_train_encoded = X_train_encoded.fillna(0)
X_test_encoded = X_test_encoded.fillna(0)

In [15]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)
joblib.dump(scaler, '../models/artifacts/scaler.pkl')

['../models/artifacts/scaler.pkl']

In [16]:
print("Data preprocessing completed!")
print(f"Training set shape: {X_train_encoded.shape}")
print(f"Test set shape: {X_test_encoded.shape}")

Data preprocessing completed!
Training set shape: (446568, 48)
Test set shape: (111643, 48)


In [17]:
models = {
    'XGBoost': XGBClassifier(
        n_estimators=2000,
        max_depth=8,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        scale_pos_weight=1.62
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=2000,
        max_depth=8,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
        class_weight={0: 0.81, 1: 1.31}
    ),
    'CatBoost': CatBoostClassifier(
        iterations=2000,
        depth=8,
        learning_rate=0.03,
        subsample=0.8,
        random_seed=42,
        verbose=False,
        thread_count=-1,
        class_weights=[0.81, 1.31]
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),
    'LogisticRegression': LogisticRegression(
        max_iter=1000,
        random_state=42,
        n_jobs=-1
    ),
    'SVM': LinearSVC(
        max_iter=2000,
        random_state=42
    )
}

In [21]:
def train_and_evaluate(model_name, model, X_train_data, X_test_data, y_train_data, y_test_data):
    print(f"\nTraining {model_name}...")

    if model_name in ['LogisticRegression', 'SVM']:
        X_train_data = X_train_scaled
        X_test_data = X_test_scaled

    if model_name == 'XGBoost':
        model.fit(
            X_train_data, y_train_data,
            eval_set=[(X_test_data, y_test_data)],
            verbose=False
        )
    elif model_name == 'LightGBM':
        model.fit(
            X_train_data, y_train_data,
            eval_set=[(X_test_data, y_test_data)],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
        )
    elif model_name == 'CatBoost':
        model.fit(
            X_train_data, y_train_data,
            eval_set=(X_test_data, y_test_data),
            early_stopping_rounds=50,
            verbose=False
        )
    else:
        model.fit(X_train_data, y_train_data)

    if hasattr(model, 'predict_proba'):
        y_pred_proba = model.predict_proba(X_test_data)[:, 1]
    else:
        y_pred_proba = model.decision_function(X_test_data)

    y_pred = model.predict(X_test_data)

    # Calculate metrics
    accuracy = accuracy_score(y_test_data, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test_data, y_pred, average='binary')
    auc_roc = roc_auc_score(y_test_data, y_pred_proba)

    return {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc_roc': auc_roc,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }


In [18]:
results = {}

In [22]:
for model_name, model in tqdm(models.items(), desc="Training models"):
    try:
        result = train_and_evaluate(
            model_name, model,
            X_train_encoded, X_test_encoded,
            y_train_encoded, y_test_encoded
        )
        results[model_name] = result
        print(f"{model_name} completed successfully!")
    except Exception as e:
        print(f"Error training {model_name}: {str(e)}")
        continue

Training models:   0%|          | 0/6 [00:00<?, ?it/s]


Training XGBoost...


Training models:  17%|█▋        | 1/6 [00:59<04:59, 59.95s/it]

XGBoost completed successfully!

Training LightGBM...
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's binary_logloss: 0.358294


Training models:  33%|███▎      | 2/6 [01:35<03:01, 45.48s/it]

...

RandomForest completed successfully!

Training LogisticRegression...


Training models:  83%|████████▎ | 5/6 [04:56<00:47, 47.73s/it]

LogisticRegression completed successfully!

Training SVM...


Training models: 100%|██████████| 6/6 [05:04<00:00, 50.76s/it]

SVM completed successfully!


In [23]:
print("\n" + "="*80)
print("MODEL COMPARISON RESULTS")
print("="*80)

comparison_df = pd.DataFrame()
for model_name, result in results.items():
    comparison_df = pd.concat([comparison_df, pd.DataFrame({
        'Model': [model_name],
        'Accuracy': [f"{result['accuracy']:.4f}"],
        'Precision': [f"{result['precision']:.4f}"],
        'Recall': [f"{result['recall']:.4f}"],
        'F1-Score': [f"{result['f1']:.4f}"],
        'AUC-ROC': [f"{result['auc_roc']:.4f}"]
    })], ignore_index=True)

print(comparison_df.to_string(index=False))


MODEL COMPARISON RESULTS
             Model Accuracy Precision Recall F1-Score AUC-ROC
           XGBoost   0.8748    0.8954 0.7605   0.8224  0.9307
          LightGBM   0.8609    0.8762 0.7397   0.8022  0.9158
          CatBoost   0.8519    0.8588 0.7320   0.7903  0.9075
      RandomForest   0.7119    0.8569 0.2933   0.4370  0.7412
LogisticRegression   0.6867    0.6833 0.3318   0.4467  0.6931
               SVM   0.6889    0.6954 0.3272   0.4450  0.6928


In [27]:
print(f"\n{'='*50}")
print("SAVING ALL TRAINED MODELS...")
print(f"{'='*50}")

for model_name, result in tqdm(results.items(), desc="Saving models"):
    model_filename = f'../models/trained_models/{model_name.lower()}_model.pkl'
    joblib.dump(result['model'], model_filename)
    print(f"{model_name} saved as {model_filename}")

print("\nAll models saved successfully!")


SAVING ALL TRAINED MODELS...


Saving models:   0%|          | 0/6 [00:00<?, ?it/s]

XGBoost saved as ../models/trained_models/xgboost_model.pkl

Saving models:  33%|███▎      | 2/6 [00:00<00:00,  5.85it/s]


LightGBM saved as ../models/trained_models/lightgbm_model.pkl


Saving models: 100%|██████████| 6/6 [00:00<00:00,  9.06it/s]

CatBoost saved as ../models/trained_models/catboost_model.pkl
RandomForest saved as ../models/trained_models/randomforest_model.pkl
LogisticRegression saved as ../models/trained_models/logisticregression_model.pkl
SVM saved as ../models/trained_models/svm_model.pkl

All models saved successfully!


Testing Neural Network 

In [28]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight

# Set random seeds for reproducibility
tf.random.set_seed(42)

In [29]:
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train_encoded),
    y=y_train_encoded
)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}
print(f"Class weights: {class_weight_dict}")

Class weights: {0: 0.808028024260672, 1: 1.3116144646255785}


In [35]:
def create_neural_network(input_dim):
    model = Sequential([
        # Input layer
        Dense(512, activation='relu', input_dim=input_dim),
        BatchNormalization(),
        Dropout(0.4),

        # Hidden layers
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),

        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),

        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),

        Dense(32, activation='relu'),
        Dropout(0.4),

        # Output layer
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )

    return model

# Create the model
input_dim = X_train_scaled.shape[1]
nn_model = create_neural_network(input_dim)

print("\nNeural Network Architecture:")
nn_model.summary()


Neural Network Architecture:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 512)            │        25,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 203,521 (795.00 KB)

 Trainable params: 201,601 (787.50 KB)

 Non-trainable params: 1,920 (7.50 KB)

In [36]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        verbose=1
    )
]


In [37]:
class TrainingProgressCallback(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.pbar = None

    def on_train_begin(self, logs=None):
        self.pbar = tqdm(total=self.params['epochs'], desc="Training Neural Network")

    def on_epoch_end(self, epoch, logs=None):
        self.pbar.update(1)
        self.pbar.set_postfix({
            'loss': f"{logs.get('loss', 0):.4f}",
            'val_loss': f"{logs.get('val_loss', 0):.4f}",
            'val_accuracy': f"{logs.get('val_accuracy', 0):.4f}"
        })

    def on_train_end(self, logs=None):
        self.pbar.close()

progress_callback = TrainingProgressCallback()
callbacks.append(progress_callback)



In [ ]:
print(f"\nStarting Neural Network training...")
print(f"Training data shape: {X_train_scaled.shape}")
print(f"Validation data shape: {X_test_scaled.shape}")
history = nn_model.fit(
    X_train_scaled, y_train_encoded,
    validation_data=(X_test_scaled, y_test_encoded),
    epochs=100,
    batch_size=1024,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=0
)

print("\nNeural Network training completed!")

In [ ]:
nn_predictions_proba = nn_model.predict(X_test_scaled, verbose=0).flatten()
nn_predictions = (nn_predictions_proba > 0.5).astype(int)

# Calculate metrics
nn_accuracy = accuracy_score(y_test_encoded, nn_predictions)
nn_precision, nn_recall, nn_f1, _ = precision_recall_fscore_support(y_test_encoded, nn_predictions, average='binary')
nn_auc_roc = roc_auc_score(y_test_encoded, nn_predictions_proba)

# Add Neural Network results to comparison
nn_result = {
    'model': nn_model,
    'accuracy': nn_accuracy,
    'precision': nn_precision,
    'recall': nn_recall,
    'f1': nn_f1,
    'auc_roc': nn_auc_roc,
    'predictions': nn_predictions,
    'probabilities': nn_predictions_proba
}

results['NeuralNetwork'] = nn_result

print(f"\nNeural Network Results:")
print(f"Accuracy: {nn_accuracy:.4f}")
print(f"Precision: {nn_precision:.4f}")
print(f"Recall: {nn_recall:.4f}")
print(f"F1-Score: {nn_f1:.4f}")
print(f"AUC-ROC: {nn_auc_roc:.4f}")


Neural Network Results:
Accuracy: 0.7975
Precision: 0.7892
Recall: 0.6396
F1-Score: 0.7066
AUC-ROC: 0.8463
